# NED API client — working example

Demonstrates `energy_dashboard.NedClient` against the real NED API.

**Before running:** copy `.env.example` to `.env` at the project root and fill in `NED_API_KEY`
with a key from your [ned.nl](https://ned.nl) account.

In [3]:
from datetime import UTC, datetime

from energy_dashboard import (
    Activity,
    Classification,
    EnergyType,
    Granularity,
    NedApiError,
    NedClient,
    Point,
    UtilizationQuery,
)

client = NedClient()  # reads NED_API_KEY (and optional NED_BASE_URL) from .env

## Build a query

This is the worked example from the NED API manual: solar generation (`Point.NETHERLANDS`,
`EnergyType.SOLAR`) at 10-minute granularity for 16 November 2020.

In [4]:
query = UtilizationQuery(
    point=Point.NETHERLANDS,
    type=EnergyType.SOLAR,
    activity=Activity.PROVIDING,
    classification=Classification.CURRENT,
    granularity=Granularity.TEN_MINUTES,
    valid_from=datetime(2020, 11, 16, tzinfo=UTC),
    valid_to=datetime(2020, 11, 17, tzinfo=UTC),
)
query.to_params()

{'point': 0,
 'type': 2,
 'activity': 1,
 'classification': 2,
 'granularity': 3,
 'granularitytimezone': 1,
 'validfrom[after]': '2020-11-16',
 'validfrom[strictly_before]': '2020-11-17',
 'page': 1,
 'itemsPerPage': 144}

## Fetch a single page

In [5]:
try:
    page = await client.get_utilizations(query)
except NedApiError as exc:
    print(f"NED API call failed: {exc}")
else:
    print(f"total_items={page.total_items}, items on this page={len(page.items)}")
    for record in page.items[:5]:
        print(
            f"{record.valid_from:%Y-%m-%d %H:%M} "
            f"capacity={record.capacity:>10.0f} kW  "
            f"volume={record.volume:>8.0f} kWh  "
            f"percentage={record.percentage:.2%}"
        )

total_items=144, items on this page=144
2020-11-15 23:00 capacity=         0 kW  volume=       0 kWh  percentage=0.00%
2020-11-15 23:10 capacity=         0 kW  volume=       0 kWh  percentage=0.00%
2020-11-15 23:20 capacity=         0 kW  volume=       0 kWh  percentage=0.00%
2020-11-15 23:30 capacity=         0 kW  volume=       0 kWh  percentage=0.00%
2020-11-15 23:40 capacity=         0 kW  volume=       0 kWh  percentage=0.00%


## Walk every page

`iter_utilizations` pages through the full result set automatically.

In [6]:
try:
    records = [r async for r in client.iter_utilizations(query)]
except NedApiError as exc:
    print(f"NED API call failed: {exc}")
else:
    print(f"fetched {len(records)} records across all pages")
    if records:
        peak = max(records, key=lambda r: r.capacity)
        print(
            f"peak capacity: {peak.capacity:.0f} kW at {peak.valid_from:%Y-%m-%d %H:%M}"
        )

fetched 144 records across all pages
peak capacity: 1174285 kW at 2020-11-16 12:30


In [7]:
await client.close()